In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML
import os
from pathlib import Path

from src.core.context import state
from src.core.utils import get_url_params
from src.ui.components import debug_console
from src.ui.styles import get_app_css
from src.ui.step_builders import build_step1
from src.core.utils import locate_factor_list_files


def initialize_app():
    # Load environment variables
    from dotenv import load_dotenv
    env_path = Path('.env')
    if env_path.exists():
        load_dotenv(env_path)

    # Check if internal app mode is enabled
    state.is_internal_app = os.getenv('INTERNAL_APP', 'false').lower() == 'true'

    # Get URL parameters
    fl_id, benchmark = get_url_params('fl_id', 'benchmark')
    if fl_id:
        state.factor_list_uid = fl_id
    if benchmark:
        state.benchmark_ticker = benchmark

    # In internal app mode with fl_id, verify files exist
    if state.is_internal_app and fl_id:
        dataset_path, formulas_path, error, file_type = locate_factor_list_files(fl_id)
        if error:
            state.files_verification_error = error
            state.files_verified = False
        else:
            state.auto_dataset_path = dataset_path
            state.auto_formulas_path = formulas_path
            state.auto_dataset_file_type = file_type
            state.files_verified = True

    # load css
    display(HTML(get_app_css()))

    # logs popup in top right corner
    debug_overlay, debug_output, debug_toggle_button = debug_console()
    state.debug_output = debug_output
    state.debug_toggle_button = debug_toggle_button

    # empty containers for all steps
    state.step2_container = widgets.VBox([], layout=widgets.Layout(display='none'))
    state.step3_container = widgets.VBox([], layout=widgets.Layout(display='none'))

    state.step1_container = build_step1()

    main_container = widgets.VBox([
        state.step1_container,
        state.step2_container,
        state.step3_container,
        debug_overlay
    ])

    return main_container


main_container = initialize_app()
display(main_container)